# Feature Engineering - Master Dataset

Cilj: spojiti sve CSV fajlove u jedan master dataset i kreirati features za modeliranje.

Output: `data/processed/train_features.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')

DATA      = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

EARTHQUAKE_DATE = pd.Timestamp('2016-04-16')

## 1. Ucitavanje podataka

In [ ]:
train        = pd.read_csv(DATA + 'train.csv', parse_dates=['date'])
test         = pd.read_csv(DATA + 'test.csv',  parse_dates=['date'])
stores       = pd.read_csv(DATA + 'stores.csv')
transactions = pd.read_csv(DATA + 'transactions.csv', parse_dates=['date'])
oil          = pd.read_csv(DATA + 'oil.csv',   parse_dates=['date'])
holidays     = pd.read_csv(DATA + 'holidays_events.csv', parse_dates=['date'])

print(f'train: {train.shape}, test: {test.shape}')

## 2. Priprema pomocnih tabela

### 2a. Oil - interpolacija za vikende i praznike

In [ ]:
# Svi datumi od pocetka traina do kraja testa
all_dates = pd.date_range(
    start=train['date'].min(),
    end=test['date'].max()
)

oil_full = (
    oil
    .set_index('date')
    .reindex(all_dates)
    .interpolate(method='linear')
    .reset_index()
    .rename(columns={'index': 'date', 'dcoilwtico': 'oil_price'})
)

print('Oil missing after interpolation:', oil_full['oil_price'].isnull().sum())
oil_full.head()

### 2b. Holidays - national i local flagovi

Pravila:
- `transferred=True` -> normalan dan, ignorisati
- `type='Bridge'` -> ignorisati za is_holiday (Bridge nije pravi praznik)
- National praznici vaze za sve prodavnice
- Local/Regional praznici vaze samo za prodavnice u tom gradu/regionu

In [ ]:
real_hol = holidays[
    (holidays['transferred'] == False) &
    (~holidays['type'].isin(['Bridge', 'Work Day']))
].copy()

# National holidays - vaze za sve
national_hol = (
    real_hol[real_hol['locale'] == 'National']
    [['date', 'type']]
    .drop_duplicates('date')
    .assign(is_national_holiday=1)
    [['date', 'is_national_holiday']]
)

# Local holidays - po gradu
local_hol = (
    real_hol[real_hol['locale'].isin(['Local', 'Regional'])]
    [['date', 'locale_name']]
    .drop_duplicates()
    .assign(is_local_holiday=1)
    .rename(columns={'locale_name': 'city'})
)

print('National holiday dates:', len(national_hol))
print('Local holiday records:', len(local_hol))
national_hol.head()

## 3. Merge - master dataset

In [ ]:
# Spajamo train i test da features kreiramo zajedno
test['sales'] = np.nan
df = pd.concat([train, test], ignore_index=True)

# + stores metadata
df = df.merge(stores, on='store_nbr', how='left')

# + transactions
df = df.merge(transactions, on=['date', 'store_nbr'], how='left')

# + oil
df = df.merge(oil_full, on='date', how='left')

# + national holidays
df = df.merge(national_hol, on='date', how='left')
df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)

# + local holidays (join na date + city prodavnice)
df = df.merge(local_hol, on=['date', 'city'], how='left')
df['is_local_holiday'] = df['is_local_holiday'].fillna(0).astype(int)

df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print(f'Master shape: {df.shape}')
print(f'Kolone: {list(df.columns)}')

## 4. Kalendarske features

In [ ]:
df['dayofweek']  = df['date'].dt.dayofweek          # 0=Mon, 6=Sun
df['month']      = df['date'].dt.month
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['dayofmonth'] = df['date'].dt.day
df['year']       = df['date'].dt.year
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# Payday: 15. i zadnji dan u mjesecu
df['days_in_month'] = df['date'].dt.days_in_month
df['is_payday'] = (
    (df['dayofmonth'] == 15) | (df['dayofmonth'] == df['days_in_month'])
).astype(int)
df.drop(columns=['days_in_month'], inplace=True)

print('Kalendarske features dodane.')

## 5. Earthquake feature

In [ ]:
# Broj dana nakon zemljotresa (0 prije, 1-30 aktivni period, 0 nakon 30 dana)
days_after = (df['date'] - EARTHQUAKE_DATE).dt.days
df['days_after_earthquake'] = days_after.clip(lower=0, upper=30)
df['days_after_earthquake'] = np.where(days_after < 0, 0, df['days_after_earthquake'])

print('Earthquake feature:')
print(df['days_after_earthquake'].value_counts().sort_index().tail(10))

## 6. Lag i rolling features

Lagi se racunaju po grupi (store_nbr, family) jer svaki red predstavlja jednu kombinaciju.

Minimum lag koji mozemo koristiti za test set (15 dana): lag_16.
Koristimo lag_7, lag_14, lag_28, lag_56 - za test set cemo popuniti lag_7 i lag_14 iz poznatih vrijednosti.

In [ ]:
# log1p transformacija PRIJE racunanja laga (da lagi budu u istom prostoru)
df['sales_log'] = np.log1p(df['sales'])

LAG_DAYS = [7, 14, 28, 56]
ROLL_WINDOWS = [7, 14, 28]

def add_lags_and_rolling(df):
    df = df.sort_values('date')
    for lag in LAG_DAYS:
        df[f'lag_{lag}'] = df['sales_log'].shift(lag)
    for window in ROLL_WINDOWS:
        df[f'roll_mean_{window}'] = df['sales_log'].shift(1).rolling(window).mean()
        df[f'roll_std_{window}']  = df['sales_log'].shift(1).rolling(window).std()
    return df

df = (
    df
    .groupby(['store_nbr', 'family'], group_keys=False)
    .apply(add_lags_and_rolling)
)

lag_cols = [f'lag_{l}' for l in LAG_DAYS] + \
           [f'roll_mean_{w}' for w in ROLL_WINDOWS] + \
           [f'roll_std_{w}' for w in ROLL_WINDOWS]

print('Lag/rolling features dodane.')
print('Missing u lag_7 (ocekivano na pocetku):', df['lag_7'].isnull().sum())

## 7. Encoding kategorijskih varijabli

In [ ]:
# Label encoding za family i store type
df['family_enc'] = df['family'].astype('category').cat.codes
df['type_enc']   = df['type'].astype('category').cat.codes
df['city_enc']   = df['city'].astype('category').cat.codes
df['state_enc']  = df['state'].astype('category').cat.codes

# Sacuvaj mappinge za kasniju interpretaciju
family_map = dict(enumerate(df['family'].astype('category').cat.categories))
type_map   = dict(enumerate(df['type'].astype('category').cat.categories))
print('Family mapping:', family_map)
print('Type mapping:', type_map)

## 8. Pregled finalnog dataseta

In [ ]:
feature_cols = [
    # identifikatori
    'date', 'store_nbr', 'family',
    # target
    'sales', 'sales_log',
    # store info
    'city', 'state', 'type', 'cluster',
    'family_enc', 'type_enc', 'city_enc', 'state_enc',
    # promocija
    'onpromotion',
    # transakcije
    'transactions',
    # nafta
    'oil_price',
    # kalendar
    'year', 'month', 'weekofyear', 'dayofweek', 'dayofmonth',
    'is_weekend', 'is_payday',
    # praznici
    'is_national_holiday', 'is_local_holiday',
    # posebni eventi
    'days_after_earthquake',
    # lagi i rolling
] + lag_cols

final = df[feature_cols].copy()

print(f'Finalni dataset: {final.shape}')
print(f'\nMissing values po koloni:')
missing = final.isnull().sum()
print(missing[missing > 0])

In [ ]:
final.head(10)

In [ ]:
# Distribucija sales vs log1p(sales)
train_only = final[final['sales'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_only['sales'].clip(upper=2000), bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Distribucija sales (clipped na 2000)')
axes[0].set_xlabel('sales')

axes[1].hist(train_only['sales_log'], bins=80, color='darkorange', edgecolor='none')
axes[1].set_title('Distribucija log1p(sales)')
axes[1].set_xlabel('log1p(sales)')

plt.tight_layout()
plt.show()

## 9. Snimanje na disk

In [ ]:
# Train i test odvojeno
train_final = final[final['sales'].notna()].copy()
test_final  = final[final['sales'].isna()].copy()

# Izbaci redove gdje lagi nisu dostupni (prvih ~56 dana po grupi)
train_clean = train_final.dropna(subset=['lag_56'])

print(f'Train before dropna: {train_final.shape}')
print(f'Train after dropna:  {train_clean.shape}')
print(f'Test:                {test_final.shape}')

train_clean.to_parquet(PROCESSED + 'train_features.parquet', index=False)
test_final.to_parquet(PROCESSED  + 'test_features.parquet',  index=False)

# Sacuvaj i mappinge
import json
with open(PROCESSED + 'encodings.json', 'w') as f:
    json.dump({'family': family_map, 'type': type_map}, f, indent=2)

print('\nSnimljeno:')
print(f'  {PROCESSED}train_features.parquet')
print(f'  {PROCESSED}test_features.parquet')
print(f'  {PROCESSED}encodings.json')

## 10. Korelacijska matrica features

Brza provjera korelacije numerickih features sa targetom.

In [ ]:
num_cols = [
    'sales_log', 'onpromotion', 'oil_price', 'transactions',
    'is_national_holiday', 'is_local_holiday', 'is_payday', 'is_weekend',
    'days_after_earthquake', 'cluster',
    'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_mean_28'
]

corr = train_clean[num_cols].corr()['sales_log'].drop('sales_log').sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
corr.plot(kind='barh', ax=ax, color=corr.map(lambda x: 'steelblue' if x > 0 else 'salmon'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Korelacija features sa log1p(sales)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

print(corr.to_string())